# Male CNS v1.0 — オスの中枢神経系まるごと

hemibrain や FlyWire との決定的な違いは **脳と腹部神経索(VNC)が繋がっている** こと。
「見る → 判断する → 脚を動かす」を1本の配線図の上で端から端まで辿れる。

- 21万ボディ / 1億5185万接続 / 3億1183万シナプス
- 下行ニューロン(脳→VNC) 1,314本、上行ニューロン 1,846本、運動ニューロン 815本
- FlyWire(メス脳)との対応付けが 143,156 件 → オスとメスの比較ができる

In [1]:
import pandas as pd
import networkx as nx

D = "../data/malecns"
ann = pd.read_feather(f"{D}/body-annotations-male-cns-v1.0-minconf-0.5.feather")
w = pd.read_feather(f"{D}/connectome-weights-male-cns-v1.0-minconf-0.5.feather")

name = ann.set_index("bodyId")["type"]
sc = ann.set_index("bodyId")["superclass"]
print(f"{len(ann):,} ボディ / {len(w):,} 接続 / {w['weight'].sum():,} シナプス")
ann["superclass"].value_counts().head(12)

211,577 ボディ / 151,856,684 接続 / 311,833,243 シナプス


superclass
ol_intrinsic          89403
cb_intrinsic          32164
vnc_intrinsic         13161
visual_projection      9201
vnc_sensory            6370
ol_sensory             6098
cb_sensory             4868
ascending_neuron       1846
descending_neuron      1314
vnc_motor               708
visual_centrifugal      563
sensory_ascending       537
Name: count, dtype: int64

## 1. 逃避回路 — 教科書の配線をデータで確かめる

ハエに手を伸ばすと飛んで逃げる。あの回路は物理的に追跡できる:

**迫ってくる影を検出する視覚ニューロン (LC4 / LPLC2) → 巨大繊維 (DNp01 / Giant Fiber) → 中脚を蹴り出す運動ニューロン (TTMn)**

DNp01 は脳の中で一番太い軸索を持つニューロンで、1本で首を貫いてVNCまで降りている。

In [2]:
gf = int(ann[ann["type"] == "DNp01"]["bodyId"].iloc[0])
print("Giant Fiber bodyId:", gf)

up = w[w["body_post"] == gf].copy()
up["partner"] = up["body_pre"].map(name)
up["sc"] = up["body_pre"].map(sc)
up.groupby(["partner", "sc"])["weight"].sum().nlargest(10)

Giant Fiber bodyId: 10001


partner  sc               
LC4      visual_projection    2580
LPLC2    visual_projection    2220
PVLP122  cb_intrinsic          614
DNp70    descending_neuron     597
SAD064   cb_intrinsic          548
SAD073   cb_intrinsic          469
PVLP010  cb_intrinsic          297
PVLP123  cb_intrinsic          296
CB1638   cb_intrinsic          270
PVLP151  cb_intrinsic          269
Name: weight, dtype: int64

上流の1位・2位が LC4 と LPLC2 = 迫ってくる物体(looming)の検出器。ここは視覚。次に下流を見る。

In [3]:
down = w[w["body_pre"] == gf].copy()
down["partner"] = down["body_post"].map(name)
down["sc"] = down["body_post"].map(sc)
down.groupby(["partner", "sc"])["weight"].sum().nlargest(10)

partner   sc              
TTMn      vnc_motor           70
GFC2      vnc_intrinsic       58
IN06B008  vnc_intrinsic       52
GFC4      vnc_intrinsic       47
GFC3      vnc_intrinsic       46
IN11A001  vnc_intrinsic       44
AN19A018  ascending_neuron    39
IN18B034  vnc_intrinsic       30
IN08B003  vnc_intrinsic       26
IN18B031  vnc_intrinsic       26
Name: weight, dtype: int64

## 2. 脳から筋肉まで、何シナプスで届くか

weight>=10 の接続だけでグラフを組んで、Giant Fiber から運動ニューロンまでの最短経路を出す。

In [4]:
strong = w[w["weight"] >= 10]
G = nx.from_pandas_edgelist(strong, "body_pre", "body_post",
                            edge_attr="weight", create_using=nx.DiGraph)
print(G)

motor = set(ann[ann["superclass"].isin(["vnc_motor", "cb_motor"])]["bodyId"])
dist = nx.single_source_shortest_path_length(G, gf, cutoff=4)
reached = {b: d for b, d in dist.items() if b in motor}
print(f"4ステップ以内に届く運動ニューロン: {len(reached)} / {len(motor)}")

target = min(reached, key=reached.get)
for i, b in enumerate(nx.shortest_path(G, gf, target)):
    print(("    " if i == 0 else " -> ") + f"{b}  {name.get(b)}  [{sc.get(b)}]")

DiGraph with 198263 nodes and 2799910 edges


4ステップ以内に届く運動ニューロン: 774 / 815
    10001  DNp01  [descending_neuron]
 -> 800146  TTMn  [vnc_motor]


TTMn (tergotrochanteral motor neuron) は中脚を伸ばして体を空中に打ち上げる筋肉を動かす。
**単シナプス** で繋がっている = 最速で逃げるための配線。数十年の電気生理の結論が、そのまま出てくる。

## 3. オス特有の回路 — 求愛

pC1 (メスでの P1 に相当) はオスの求愛行動の指令ニューロン。ここを刺激するとオスは
一人でも求愛の歌を歌い始める。性的二型のラベルが付いているものを見る。

In [5]:
pc1 = ann[ann["type"].fillna("").str.startswith("pC1")]
print(f"pC1 系: {len(pc1)} ボディ / {pc1['type'].nunique()} 亜型")
pc1[["bodyId", "type", "somaSide", "dimorphism", "fruDsx"]].head(12)

pC1 系: 156 ボディ / 49 亜型


,bodyId,type,somaSide,dimorphism,fruDsx
201,10217,pC1x_b,L,sexually dimorphic,dsx_high
625,10666,pC1x_c,R,sexually dimorphic,dsx_high
2132,12257,pC1x_c,L,sexually dimorphic,dsx_high
2234,12366,pC1x_a,R,sexually dimorphic,dsx_high
2308,12442,pC1_4a,L,male-specific,dsx_high
2313,12448,pC1_18a,L,male-specific,dsx_low
2736,12893,pC1x_d,L,sexually dimorphic,dsx_high
3089,13266,pC1_3b,L,male-specific,coexpress_high
3262,13449,pC1x_d,R,sexually dimorphic,dsx_high
3304,13495,pC1_11a,R,male-specific,dsx_high


In [6]:
pc1_ids = set(pc1["bodyId"])
d = w[w["body_pre"].isin(pc1_ids)].copy()
d["partner"] = d["body_post"].map(name)
d["sc"] = d["body_post"].map(sc)
d.groupby(["partner", "sc"])["weight"].sum().nlargest(20)

partner   sc               
pC1_18b   cb_intrinsic         4516
AVLP749m  cb_intrinsic         3752
SIP104m   cb_intrinsic         3722
SIP106m   cb_intrinsic         3541
mAL_m8    cb_intrinsic         3523
SIP122m   cb_intrinsic         3173
AVLP717m  cb_intrinsic         3118
SIP121m   cb_intrinsic         3068
aIPg5     cb_intrinsic         2909
SIP141m   cb_intrinsic         2825
SIP133m   cb_intrinsic         2625
SIP146m   cb_intrinsic         2322
pC1x_d    cb_intrinsic         2215
pC1_4a    cb_intrinsic         2128
CL344_b   cb_intrinsic         2100
SMP702m   cb_intrinsic         2094
SIP103m   cb_intrinsic         1966
AVLP705m  cb_intrinsic         1960
pIP10     descending_neuron    1941
aIPg_m1   cb_intrinsic         1913
Name: weight, dtype: int64

## 4. 性的二型のニューロンを数える

In [7]:
ann["dimorphism"].value_counts()

dimorphism
male-specific                     1258
sexually dimorphic                 771
potentially sexually dimorphic     177
potentially male-specific          162
Name: count, dtype: int64

In [8]:
ms = ann[ann["dimorphism"].fillna("").str.contains("male-specific")]
ms["type"].value_counts().head(20)

type
Tm26        103
mAL_m8       16
SMP703m      14
PVLP209m     13
LoVP92       13
Cm26         13
AVLP753m     12
VES200m      12
AVLP749m     12
mAL_m1       12
FLA002m      12
FLA001m      12
AVLP743m     11
SCL001m      11
LH008m       11
SAD200m      11
SMP723m      11
FLA004m      11
mAL_m3c      10
SCL002m      10
Name: count, dtype: int64

## 5. メス(FlyWire)との対応付け

`flywireType` 列にメス脳での型名が入っている。ここを鍵にすると
`data/flywire/flywire_783_annotations.tsv` と突き合わせて、
同じ細胞型がオスとメスで何個ずつあるか比べられる。

In [9]:
fw = pd.read_csv("../data/flywire/flywire_783_annotations.tsv", sep="	", low_memory=False)

m_counts = ann["flywireType"].value_counts().rename("male")
f_counts = fw["cell_type"].value_counts().rename("female")
cmp = pd.concat([m_counts, f_counts], axis=1).dropna()
cmp["ratio"] = cmp["male"] / cmp["female"]
cmp[cmp["female"] >= 4].sort_values("ratio", ascending=False).head(20)

,male,female,ratio
Tm40,180.0,41.0,4.390244
LC28b,61.0,16.0,3.812500
Sm43,28.0,11.0,2.545455
LCe08,12.0,5.0,2.400000
VC5_lvPN,12.0,5.0,2.400000
CB1852,9.0,4.0,2.250000
CB1786_c,9.0,4.0,2.250000
Sm27,31.0,14.0,2.214286
AN_GNG_182,11.0,5.0,2.200000
CB3017,8.0,4.0,2.000000


## 次にやると面白いこと

- 上行ニューロン(`ascending_neuron`)を辿って「脚からの感覚が脳のどこに届くか」
- `body-neurotransmitters-male-cns-v1.0.feather` を繋いで興奮性/抑制性で経路を色分け
- 特定の脚の運動ニューロンから上流に遡って「歩行を作る回路」を切り出す
- https://neuprint.janelia.org / https://male-cns.janelia.org で bodyId を検索して3D表示